# 词袋模型（Bag-of-Words）

特征工程：把文本变成稀疏计数向量。与 Word2Vec 对照——这里是**手工表示**，下一篇 Skip-gram 用神经网络学**稠密词向量**。

语料与 `14_word2vec_skip_gram` 相同，便于串联阅读。


In [ ]:
import numpy as np
from itertools import chain
from collections import Counter

corpus = [
    "我 喜欢 学习 机器 学习",
    "机器 学习 很 有趣",
    "我 喜欢 编程",
]

def tokenize(sentences):
    return [s.split() for s in sentences]

docs = tokenize(corpus)
vocab = sorted(set(chain.from_iterable(docs)))
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for w, i in w2i.items()}
print("词表 |V| =", len(vocab))
print(vocab)


## 1. 文档–词计数矩阵（BoW）

每一行是一篇文档，每一列是词表中的一个词，值为该词在文档中出现的次数。


In [ ]:
def bow_vector(tokens, w2i):
    v = np.zeros(len(w2i), dtype=int)
    for t in tokens:
        v[w2i[t]] += 1
    return v

X_bow = np.vstack([bow_vector(d, w2i) for d in docs])
print("形状:", X_bow.shape, "→ (文档数, |V|)")
print("列 =", vocab)
print(X_bow)


## 2. 与 one-hot 的关系

- **词级 one-hot**：单个词 → 长度为 \(|V|\) 的向量，仅对应位置为 1。
- **文档 BoW**：可看成文档内各词 one-hot 的**求和**（或多重集合计数）；仍稀疏、维度 = \(|V|\)，且**丢掉语序**。


In [ ]:
def one_hot(word, w2i):
    v = np.zeros(len(w2i), dtype=int)
    v[w2i[word]] = 1
    return v

# 文档0 的 BoW = 各词 one-hot 之和
doc0 = docs[0]
sum_oh = sum(one_hot(w, w2i) for w in doc0)
print("文档0 tokens:", doc0)
print("one-hot 求和:", sum_oh)
print("BoW 向量:   ", X_bow[0])
print("一致?", np.array_equal(sum_oh, X_bow[0]))

print("\n词「学习」的 one-hot:", one_hot("学习", w2i))


## 3. 过渡到 Word2Vec

BoW / one-hot：**稀疏、无序、无语义相似度**（「喜欢」与「有趣」正交）。  
下一篇 Skip-gram：用上下文预测学稠密向量，使语义相近的词在向量空间靠近。
